[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module1/06-functions.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module1/06-functions.ipynb)

# Lesson 6 — Functions

**Module 1 — Python Fundamentals** | ⏱ 25 min

Functions are the primary building blocks of reusable code. A function is a named block of code that performs a specific task and can be called (invoked) whenever that task is needed. Functions reduce repetition, make code easier to test, and help you think in terms of well-defined operations. Python functions are flexible: they support default arguments, keyword arguments, variable-length argument lists, and can even return multiple values.

## Learning Objectives
- Define functions using `def` and call them with arguments
- Distinguish positional, keyword, and default arguments
- Use `*args` and `**kwargs` for variable-length argument lists
- Return single and multiple values from functions
- Understand local vs global scope and closures
- Write anonymous functions with `lambda`
- Document functions with docstrings

## Defining and Calling Functions

A function is defined with the `def` keyword, followed by the function name, parentheses containing any parameters, and a colon. The function body is indented. Functions should do one thing and do it well — this is the **single responsibility principle**. A function that does not have a `return` statement implicitly returns `None`. Python conventions (PEP 8) use `snake_case` for function names.

In [ ]:
# Defining a simple function
def greet(name):
    """Return a personalised greeting message."""
    return f"Hello, {name}! Welcome to Python."

# Calling the function
message = greet("Alice")
print(message)          # Hello, Alice! Welcome to Python.
print(greet("Bob"))     # Hello, Bob! Welcome to Python.

# Functions without a return statement return None
def print_divider(width=40):
    print("-" * width)

result = print_divider()  # Prints the divider
print(result)              # None

## Positional, Keyword, and Default Arguments

Python functions support three kinds of arguments. **Positional arguments** are matched by position (the order you pass them). **Keyword arguments** are passed by name and can appear in any order. **Default arguments** specify a value used when the caller does not provide one. A critical rule: default arguments must come after non-default arguments. Also, never use mutable objects (like lists or dicts) as default argument values — this is a famous Python gotcha.

In [ ]:
# Positional and keyword arguments
def create_profile(username, age, role="user", active=True):
    """Create a user profile dictionary."""
    return {
        "username": username,
        "age": age,
        "role": role,
        "active": active
    }

# All positional
profile1 = create_profile("alice", 28)
print(profile1)

# Mix of positional and keyword
profile2 = create_profile("bob", 35, role="admin")
print(profile2)

# All keyword (order does not matter)
profile3 = create_profile(active=False, age=22, username="carol", role="moderator")
print(profile3)

In [ ]:
# THE MUTABLE DEFAULT ARGUMENT TRAP — a classic Python gotcha!

# BAD: list is created ONCE when the function is defined, not on each call
def add_item_bad(item, cart=[]):
    cart.append(item)
    return cart

print(add_item_bad("apple"))   # ['apple']
print(add_item_bad("banana"))  # ['apple', 'banana'] — the list PERSISTS!
print(add_item_bad("cherry"))  # ['apple', 'banana', 'cherry'] — bug!

# GOOD: use None as default and create the list inside the function
def add_item_good(item, cart=None):
    if cart is None:
        cart = []  # Create a fresh list on each call
    cart.append(item)
    return cart

print()
print(add_item_good("apple"))   # ['apple']
print(add_item_good("banana"))  # ['banana'] — fresh list each time

## *args and **kwargs

`*args` allows a function to accept any number of **positional** arguments, collected into a tuple. `**kwargs` allows any number of **keyword** arguments, collected into a dictionary. The names `args` and `kwargs` are conventions — the `*` and `**` syntax is what matters. These are useful when writing wrapper functions, decorators, or any function that needs to pass arguments through to another function.

In [ ]:
# *args — variable number of positional arguments
def calculate_total(*prices):
    """Sum any number of prices."""
    print(f"Prices received: {prices}")  # prices is a tuple
    return sum(prices)

print(calculate_total(10.99))                       # 10.99
print(calculate_total(10.99, 25.50, 5.00))          # 41.49
print(calculate_total(1, 2, 3, 4, 5, 6, 7, 8, 9))  # 45

# Unpacking a list with * when calling
item_prices = [12.50, 8.99, 3.25]
print(calculate_total(*item_prices))  # Unpacks the list as separate arguments

In [ ]:
# **kwargs — variable number of keyword arguments
def build_html_tag(tag, content, **attributes):
    """Build an HTML tag string with any number of attributes."""
    attr_str = ""
    for key, value in attributes.items():
        attr_str += f' {key}="{value}"'
    return f"<{tag}{attr_str}>{content}</{tag}>"

print(build_html_tag("p", "Hello world"))
print(build_html_tag("a", "Click me", href="/home", class_="btn"))
print(build_html_tag("img", "", src="photo.jpg", alt="Profile photo", width="100"))

# Combining *args and **kwargs — the most flexible signature
def log_event(event_name, *details, severity="INFO", **metadata):
    """Log an event with optional details and metadata."""
    detail_str = ", ".join(str(d) for d in details)
    meta_str = ", ".join(f"{k}={v}" for k, v in metadata.items())
    print(f"[{severity}] {event_name}: {detail_str} | {meta_str}")

log_event("user_login", "alice", "192.168.1.1", severity="INFO", session_id="abc123")
log_event("payment_failed", "card_declined", severity="ERROR", amount=99.99, currency="USD")

## Return Values

Functions can return a single value, multiple values (as a tuple), or nothing at all. Returning multiple values as a tuple is a common Python pattern — the caller can unpack the tuple directly. Functions can also return early with `return` in the middle of the body, which is useful for guard clauses. A well-designed function should have a clear, consistent return type.

In [ ]:
# Returning multiple values — Python packs them as a tuple
def analyse_numbers(data):
    """Return summary statistics for a list of numbers."""
    if not data:
        return None, None, None, None  # Guard: return None values for empty input

    minimum = min(data)
    maximum = max(data)
    average = sum(data) / len(data)
    median_idx = len(data) // 2
    sorted_data = sorted(data)
    median = sorted_data[median_idx]

    return minimum, maximum, average, median  # Returns a tuple

# Unpack the returned tuple
readings = [18, 22, 31, 15, 27, 19, 24, 29]
lo, hi, avg, med = analyse_numbers(readings)
print(f"Min: {lo}, Max: {hi}, Avg: {avg:.1f}, Median: {med}")

# Can also capture the whole tuple
stats = analyse_numbers(readings)
print(f"Type of return value: {type(stats)}")  # <class 'tuple'>

## Scope — Local vs Global

In Python, variables defined inside a function are **local** — they exist only while the function runs and cannot be accessed outside. Variables defined at the module level are **global**. If you need to modify a global variable inside a function, you must declare it with the `global` keyword. However, heavy use of global variables is considered bad practice — prefer passing values in as arguments and returning results.

In [ ]:
# Local vs global scope
api_base_url = "https://api.example.com"  # Global variable

def make_endpoint(path):
    # Reading a global variable is fine without 'global'
    full_url = api_base_url + path  # 'full_url' is local to this function
    return full_url

print(make_endpoint("/users"))   # https://api.example.com/users
# print(full_url)  # NameError: full_url is not accessible outside the function

# Modifying a global variable (avoid when possible)
request_count = 0

def track_request():
    global request_count          # Required to MODIFY the global variable
    request_count += 1
    print(f"Request #{request_count}")

track_request()  # Request #1
track_request()  # Request #2
print(f"Total requests: {request_count}")  # 2

# Better pattern: use a class or function with return values instead of global

## Closures

A **closure** is a function that remembers the variables from its enclosing scope even after the outer function has finished executing. Closures are created when an inner function references variables from the outer function. They are a powerful pattern for creating **factory functions** — functions that generate customised functions — and for maintaining state without using classes.

In [ ]:
# Closure example: counter factory
def make_counter(start=0, step=1):
    """Create a counter function that remembers its state."""
    count = [start]  # Mutable container — avoids 'nonlocal' keyword

    def counter():
        current = count[0]
        count[0] += step
        return current

    return counter  # Return the inner function — it 'closes over' count

# Create two independent counters
counter_by_1 = make_counter(start=0, step=1)
counter_by_5 = make_counter(start=0, step=5)

print(counter_by_1())  # 0
print(counter_by_1())  # 1
print(counter_by_1())  # 2
print(counter_by_5())  # 0 — independent from counter_by_1
print(counter_by_5())  # 5

# Practical closure: multiplier factory
def make_multiplier(factor):
    """Return a function that multiplies its argument by factor."""
    def multiply(value):
        return value * factor  # 'factor' is closed over from outer scope
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(7))   # 14
print(triple(7))   # 21
print(list(map(double, [1, 2, 3, 4, 5])))  # [2, 4, 6, 8, 10]

## Lambda Functions

A `lambda` is a small, anonymous function defined in a single expression. The syntax is `lambda parameters: expression`. Lambdas are commonly used as short callback functions passed to `sorted()`, `map()`, `filter()`, and similar functions. They cannot contain statements (like `if` blocks or `for` loops), only a single expression. For anything more complex, use a regular `def` function.

In [ ]:
# Lambda syntax: lambda parameters: expression
square = lambda x: x ** 2
add = lambda x, y: x + y

print(square(5))    # 25
print(add(3, 7))    # 10

# Most common use: as the key= argument for sorting
products = [
    {"name": "Keyboard",   "price": 79.99,  "rating": 4.5},
    {"name": "Monitor",    "price": 349.00, "rating": 4.8},
    {"name": "Mouse",      "price": 29.99,  "rating": 4.3},
    {"name": "Webcam",     "price": 89.99,  "rating": 4.1},
]

# Sort by price (ascending)
by_price = sorted(products, key=lambda p: p["price"])
print("By price:", [p["name"] for p in by_price])

# Sort by rating (descending)
by_rating = sorted(products, key=lambda p: p["rating"], reverse=True)
print("By rating:", [p["name"] for p in by_rating])

# Sort by multiple criteria: price ascending, then name alphabetically
by_multi = sorted(products, key=lambda p: (p["price"], p["name"]))
print("Multi-sort:", [(p["name"], p["price"]) for p in by_multi])

## Docstrings

A **docstring** is a string literal that appears as the first statement in a function, class, or module. It documents what the function does, its parameters, and its return value. Python stores docstrings in the `__doc__` attribute and tools like IDEs, `help()`, and documentation generators use them. Well-written docstrings are an essential professional practice.

In [ ]:
def calculate_compound_interest(principal, annual_rate, years, compounds_per_year=12):
    """
    Calculate compound interest and return the final amount.

    Args:
        principal (float): Initial investment amount in dollars.
        annual_rate (float): Annual interest rate as a decimal (e.g., 0.05 for 5%).
        years (int): Number of years to compound.
        compounds_per_year (int): How many times interest compounds per year.
                                  Default is 12 (monthly).

    Returns:
        float: The final amount after compounding.

    Examples:
        >>> calculate_compound_interest(1000, 0.05, 10)
        1647.0094...
        >>> calculate_compound_interest(1000, 0.05, 10, 1)  # Annual compounding
        1628.894...
    """
    n = compounds_per_year
    r = annual_rate
    t = years
    amount = principal * (1 + r / n) ** (n * t)
    return amount

# Access the docstring
print(calculate_compound_interest.__doc__[:120])  # First 120 chars

# Test the function
initial = 5000
final = calculate_compound_interest(initial, 0.06, 20)
print(f"\n${initial:,} invested at 6% for 20 years = ${final:,.2f}")
# $5,000 invested at 6% for 20 years = $16,310.19

## Practice Exercises

1. Write a function `celsius_to_all(temp_c)` that returns a dictionary with the temperature converted to Fahrenheit, Kelvin, and Rankine. Unpack and print all four values in a formatted output.
2. Write a function `flatten(*lists)` that accepts any number of lists and returns a single flat list containing all elements. Test it with `flatten([1,2], [3,4,5], [6])`.
3. Create a closure called `make_validator(min_val, max_val)` that returns a function. The returned function should accept a number and return `True` if it is within `[min_val, max_val]`, or raise a `ValueError` with a descriptive message otherwise.
4. Use `lambda` and `sorted()` to sort this list of tuples by the second element descending, then by the first element ascending: `[("charlie", 85), ("alice", 92), ("bob", 85), ("diana", 92)]`.